# Аномалии ИНН за февраль (Excel vs ODS SA-active)

Проверяем два правила:
1. У каждого уникального ИНН должен быть один договор.
2. Количество уникальных ИНН должно совпадать с количеством уникальных наименований.

Источники:
- Excel: `02_Февраль_2026.xlsx`
- ODS: `ods_alpha.scd1_agreements` + `ods_alpha.scd1_companies` (SA active на `2026-02-01`).

In [ ]:
import re
from decimal import Decimal, InvalidOperation
import pandas as pd
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Параметры
excel_path = '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx'
month_start = '2026-02-01'

excel_inn_col = 'ИНН'
excel_contract_col = 'Номер договора'
excel_name_col = 'Наименование'

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()
print('Impala connection initialized')

In [ ]:
def normalize_id(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    s = s.replace(',', '.')
    if re.search(r"[eE][+-]?\d+$", s):
        try:
            s = format(Decimal(s), 'f')
        except InvalidOperation:
            return None
    s = re.sub(r"\.0+$", '', s)
    return s if re.fullmatch(r"\d+", s) else None

def normalize_text(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', ' ')
    s = re.sub(r"\s+", ' ', s)
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    return s

def detect_excel_header_row(path, required_cols, scan_rows=30):
    raw = pd.read_excel(path, header=None)
    req = set([str(c).strip() for c in required_cols])
    max_rows = min(scan_rows, len(raw))
    for i in range(max_rows):
        vals = set([str(x).strip() for x in raw.iloc[i].tolist()])
        if len(req & vals) >= 2:
            return i
    return 0

## 0) Загрузка данных Excel и ODS в единый формат

In [ ]:
# Excel
header_row = detect_excel_header_row(excel_path, [excel_inn_col, excel_contract_col, excel_name_col])
df_excel_raw = pd.read_excel(excel_path, header=header_row)
df_excel_raw.columns = [str(c).strip() for c in df_excel_raw.columns]

for c in [excel_inn_col, excel_contract_col, excel_name_col]:
    if c not in df_excel_raw.columns:
        raise ValueError(f'В Excel отсутствует колонка: {c}')

excel_df = pd.DataFrame({
    'inn': df_excel_raw[excel_inn_col].apply(normalize_id),
    'contract_number': df_excel_raw[excel_contract_col].apply(normalize_text),
    'client_name': df_excel_raw[excel_name_col].apply(normalize_text),
})
excel_df['source'] = 'excel'
excel_df = excel_df.dropna(subset=['inn']).drop_duplicates()

# ODS (SA active на 01.02)
with imp:
    ods_raw = imp.fetch(f"""
        select
            cast(c.c_inn as string) as inn,
            cast(a.c_agr_number as string) as contract_number,
            cast(c.c_cmp_name as string) as client_name
        from ods_alpha.scd1_agreements a
        join ods_alpha.scd1_companies c
          on c.n_cmp = a.n_cmp_client
        where a.acq_class = 'SA'
          and cast(a.d_valid_from as date) <= cast('{month_start}' as date)
          and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
          and c.c_inn is not null
    """)

ods_df = pd.DataFrame({
    'inn': ods_raw['inn'].apply(normalize_id),
    'contract_number': ods_raw['contract_number'].apply(normalize_text),
    'client_name': ods_raw['client_name'].apply(normalize_text),
})
ods_df['source'] = 'ods_sa_active'
ods_df = ods_df.dropna(subset=['inn']).drop_duplicates()

display(pd.DataFrame([
    {'source': 'excel', 'header_row': header_row, 'rows': len(excel_df), 'unique_inn': excel_df['inn'].nunique()},
    {'source': 'ods_sa_active', 'header_row': None, 'rows': len(ods_df), 'unique_inn': ods_df['inn'].nunique()},
]))

## 1) Правило: у одного ИНН должен быть один договор

In [ ]:
def rule1_anomalies(df, source_name):
    z = df.dropna(subset=['inn']).copy()
    z['contract_key'] = z['contract_number'].fillna('__NULL__')
    g = (
        z.groupby('inn', as_index=False)
         .agg(contract_cnt=('contract_key', 'nunique'))
    )
    bad = g[g['contract_cnt'] > 1].copy()
    details = z[z['inn'].isin(bad['inn'])][['inn', 'contract_number', 'client_name']].drop_duplicates()
    summary = pd.DataFrame([{
        'source': source_name,
        'unique_inn': z['inn'].nunique(),
        'inn_with_multi_contracts': bad['inn'].nunique(),
        'share_pct': round(100.0 * bad['inn'].nunique() / max(z['inn'].nunique(), 1), 2)
    }])
    return summary, bad, details

r1_excel_summary, r1_excel_bad, r1_excel_details = rule1_anomalies(excel_df, 'excel')
r1_ods_summary, r1_ods_bad, r1_ods_details = rule1_anomalies(ods_df, 'ods_sa_active')

display(pd.concat([r1_excel_summary, r1_ods_summary], ignore_index=True))
print('Excel anomalies (top 100):')
display(r1_excel_details.head(100))
print('ODS anomalies (top 100):')
display(r1_ods_details.head(100))

## 2) Правило: количество уникальных ИНН = количеству уникальных наименований

In [ ]:
def rule2_checks(df, source_name):
    z = df.copy()
    unique_inn = z['inn'].dropna().nunique()
    unique_name = z['client_name'].dropna().nunique()

    inn_to_names = (
        z.dropna(subset=['inn', 'client_name'])
         .groupby('inn', as_index=False)
         .agg(name_cnt=('client_name', 'nunique'))
    )
    inn_multi_names = inn_to_names[inn_to_names['name_cnt'] > 1]

    name_to_inn = (
        z.dropna(subset=['inn', 'client_name'])
         .groupby('client_name', as_index=False)
         .agg(inn_cnt=('inn', 'nunique'))
    )
    name_multi_inn = name_to_inn[name_to_inn['inn_cnt'] > 1]

    summary = pd.DataFrame([{
        'source': source_name,
        'unique_inn': unique_inn,
        'unique_client_name': unique_name,
        'delta_inn_minus_name': unique_inn - unique_name,
        'inn_with_multi_names': len(inn_multi_names),
        'names_with_multi_inn': len(name_multi_inn),
    }])

    inn_multi_names_details = z[z['inn'].isin(inn_multi_names['inn'])][['inn', 'client_name', 'contract_number']].drop_duplicates()
    name_multi_inn_details = z[z['client_name'].isin(name_multi_inn['client_name'])][['client_name', 'inn', 'contract_number']].drop_duplicates()
    return summary, inn_multi_names_details, name_multi_inn_details

r2_excel_summary, r2_excel_inn_names, r2_excel_name_inn = rule2_checks(excel_df, 'excel')
r2_ods_summary, r2_ods_inn_names, r2_ods_name_inn = rule2_checks(ods_df, 'ods_sa_active')

display(pd.concat([r2_excel_summary, r2_ods_summary], ignore_index=True))
print('Excel: INN with multiple names (top 100):')
display(r2_excel_inn_names.head(100))
print('Excel: names with multiple INN (top 100):')
display(r2_excel_name_inn.head(100))
print('ODS: INN with multiple names (top 100):')
display(r2_ods_inn_names.head(100))
print('ODS: names with multiple INN (top 100):')
display(r2_ods_name_inn.head(100))

## 3) Финальная сводка нарушений и top-примеры

In [ ]:
final_summary = pd.concat([
    r1_excel_summary.assign(rule='rule1_inn_one_contract'),
    r1_ods_summary.assign(rule='rule1_inn_one_contract'),
    r2_excel_summary.assign(rule='rule2_unique_inn_eq_unique_name'),
    r2_ods_summary.assign(rule='rule2_unique_inn_eq_unique_name'),
], ignore_index=True, sort=False)

display(final_summary)

print('TOP examples | Excel rule1:')
display(r1_excel_details.head(30))
print('TOP examples | ODS rule1:')
display(r1_ods_details.head(30))
print('TOP examples | Excel rule2 (inn -> names):')
display(r2_excel_inn_names.head(30))
print('TOP examples | ODS rule2 (inn -> names):')
display(r2_ods_inn_names.head(30))